# 06 — End-to-End Evaluation

Computes detection metrics, end-to-end case success rates with 95% bootstrap CIs, and a Pipeline A vs B comparison.

Outputs: `detection_per_image.csv`, `summary_metrics.json`, `pipeline_comparison.png`

In [ ]:
import os
FLAG = "/content/.deps_ok_karyotype"
if not os.path.exists(FLAG):
    !pip -q install --no-cache-dir --upgrade --force-reinstall "numpy==2.0.2" "pandas==2.2.2" "opencv-python==4.10.0.84" "scikit-learn==1.5.2" "ultralytics==8.*" "torch" "torchvision" "tqdm==4.*" "matplotlib==3.*"
    open(FLAG,"w").write("ok")
    raise SystemExit("Restart runtime, then re-run.")
print("Dependencies OK")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob
from pathlib import Path
import numpy as np, pandas as pd
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt

ROOT          = "/content/drive/MyDrive/54816"
OUT_A         = f"{ROOT}/e2e_run_pipeline_A"
OUT_B         = f"{ROOT}/e2e_run_pipeline_B"
EVAL_DIR      = f"{ROOT}/e2e_eval"
os.makedirs(EVAL_DIR, exist_ok=True)

MULTI_IMG_DIR = f"{ROOT}/24_chromosomes_object/JEPG"
MULTI_XML_DIR = f"{ROOT}/24_chromosomes_object/annotations"
IOU_THR       = 0.50
SCORE_THR     = 0.25


In [ ]:
def parse_voc(xml_path):
    root=ET.parse(xml_path).getroot()
    objs=[]
    for obj in root.findall("object"):
        name=(obj.findtext("name") or "").strip()
        b=obj.find("bndbox")
        if not name or b is None: continue
        objs.append({"label":name,
                     "bbox":[float(b.findtext("xmin")),float(b.findtext("ymin")),
                              float(b.findtext("xmax")),float(b.findtext("ymax"))]})
    return objs

xml_files = glob.glob(os.path.join(MULTI_XML_DIR,"*.xml"))
gt = {}
for xp in xml_files:
    stem=Path(xp).stem
    gt[stem]=parse_voc(xp)

all_labels=sorted({o["label"] for objs in gt.values() for o in objs})
print(f"GT loaded: {len(gt)} images | {len(all_labels)} classes: {all_labels[:6]}")


In [ ]:
def iou_xyxy(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1=max(ax1,bx1); iy1=max(ay1,by1); ix2=min(ax2,bx2); iy2=min(ay2,by2)
    inter=max(0,ix2-ix1)*max(0,iy2-iy1)
    union=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/union if union>0 else 0.0

def match_preds(preds, gt_objs, iou_thr=0.50):
    matched=set(); tp=fp=0
    for p in sorted(preds, key=lambda x:-x["conf"]):
        best_iou=0; best_j=None
        for j,g in enumerate(gt_objs):
            if j in matched or g["label"]!=p["label"]: continue
            v=iou_xyxy(p["bbox"],g["bbox"])
            if v>best_iou: best_iou=v; best_j=j
        if best_iou>=iou_thr and best_j is not None: tp+=1; matched.add(best_j)
        else: fp+=1
    return tp, fp, len(gt_objs)-len(matched)


In [ ]:
import torch
from ultralytics import YOLO

YOLO24_W = f"{ROOT}/weight/best-24_chromosomes.pt"
DEVICE   = 0 if torch.cuda.is_available() else "cpu"
yolo     = YOLO(YOLO24_W)
rows=[]

for stem, gt_objs in list(gt.items()):
    candidates=list(Path(MULTI_IMG_DIR).glob(f"{stem}.*"))
    if not candidates: continue
    r=yolo(str(candidates[0]),verbose=False,device=DEVICE,conf=SCORE_THR)[0]
    id2lbl=r.names if isinstance(r.names,dict) else {i:n for i,n in enumerate(r.names)}
    preds=[]
    if r.boxes and len(r.boxes):
        for box in r.boxes:
            x1,y1,x2,y2=box.xyxy[0].tolist()
            preds.append({"label":id2lbl[int(box.cls[0])],"conf":float(box.conf[0]),"bbox":[x1,y1,x2,y2]})
    tp,fp,fn=match_preds(preds,gt_objs,IOU_THR)
    rows.append({"image":stem,"tp":tp,"fp":fp,"fn":fn,"n_gt":len(gt_objs),"n_pred":len(preds)})

df=pd.DataFrame(rows)
TP=df.tp.sum(); FP=df.fp.sum(); FN=df.fn.sum()
prec=TP/(TP+FP) if TP+FP else 0; rec=TP/(TP+FN) if TP+FN else 0; f1=2*prec*rec/(prec+rec) if prec+rec else 0
print(f"Detection (YOLO 24-class, IoU>={IOU_THR}):")
print(f"  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
df.to_csv(os.path.join(EVAL_DIR,"detection_per_image.csv"),index=False)


In [ ]:
def bootstrap_ci(vals, n_boot=1000, alpha=0.05, seed=42):
    rng=np.random.default_rng(seed); n=len(vals)
    boots=[np.mean(rng.choice(vals,n,replace=True)) for _ in range(n_boot)]
    return np.percentile(boots,[100*alpha/2,100*(1-alpha/2)])

summary={}
for label, path in [("Pipeline_A",os.path.join(OUT_A,"pipeline_a_results.json")),
                     ("Pipeline_B",os.path.join(OUT_B,"pipeline_b_results.json"))]:
    if not os.path.isfile(path):
        print(f"Not found: {path} - run NB04/NB05 first"); continue
    with open(path) as f: res=json.load(f)
    strict=np.array([int(not r["alerts"] and not any(
        v.get("anomaly",False) for v in (r.get("ae_scores") or {}).values())) for r in res])
    tol=np.array([int(len(r["alerts"])<=2) for r in res])
    ci_s=bootstrap_ci(strict); ci_t=bootstrap_ci(tol); n=len(res)
    summary[label]={"n":n,"strict_success":float(strict.mean()),
                    "strict_95ci":list(ci_s.tolist()),
                    "tolerant_success":float(tol.mean()),
                    "tolerant_95ci":list(ci_t.tolist())}
    print(f"{label} (n={n})")
    print(f"  Strict:   {strict.mean():.3f} [{ci_s[0]:.3f}-{ci_s[1]:.3f}]")
    print(f"  Tolerant: {tol.mean():.3f} [{ci_t[0]:.3f}-{ci_t[1]:.3f}]")

with open(os.path.join(EVAL_DIR,"summary_metrics.json"),"w") as f: json.dump(summary,f,indent=2)
print("Summary saved.")


In [ ]:
if summary:
    labels_plot=list(summary.keys())
    sv=[summary[l]["strict_success"] for l in labels_plot]
    tv=[summary[l]["tolerant_success"] for l in labels_plot]
    x=np.arange(len(labels_plot)); w=0.35
    fig,ax=plt.subplots(figsize=(8,5))
    ax.bar(x-w/2,sv,w,label="Strict",color="#1f77b4")
    ax.bar(x+w/2,tv,w,label="Tolerant",color="#ff7f0e")
    ax.set_xticks(x); ax.set_xticklabels(labels_plot)
    ax.set_ylabel("Fraction of cases"); ax.set_ylim(0,1.05)
    ax.set_title("Pipeline A vs B — End-to-End Success Rate")
    ax.legend(); plt.tight_layout()
    fig.savefig(os.path.join(EVAL_DIR,"pipeline_comparison.png"),dpi=150)
    plt.show(); print("Figure saved.")
